In [ ]:
"""C 2023 Matthew C. Digman
Notebook to Demonstrate Capabilities of DTMCMC on Gaussian likelihood
"""

# THE SOFTWARE IS PROVIDED "AS IS", WITHOUT WARRANTY OF ANY KIND, EXPRESS OR
# IMPLIED, INCLUDING BUT NOT LIMITED TO THE WARRANTIES OF MERCHANTABILITY,
# FITNESS FOR A PARTICULAR PURPOSE AND NONINFRINGEMENT. IN NO EVENT SHALL THE
# AUTHORS OR COPYRIGHT HOLDERS BE LIABLE FOR ANY CLAIM, DAMAGES OR OTHER
# LIABILITY, WHETHER IN AN ACTION OF CONTRACT, TORT OR OTHERWISE, ARISING FROM,
# OUT OF OR IN CONNECTION WITH THE SOFTWARE OR THE USE OR OTHER DEALINGS IN THE
# SOFTWARE.

# import needed libaries
# note: run this notebook from the repository root so that the DTMCMC
# package and diagnostic_commentary_helpers are importable
import configparser

import matplotlib.pyplot as plt
import numpy as np
from numba import njit

import diagnostic_commentary_helpers as dch
import DTMCMC.auxilliary_manager as am
import DTMCMC.de_manager as dm
import DTMCMC.exchange_manager as eh
import DTMCMC.fisher_manager as fm
import DTMCMC.prior_manager as pm
import DTMCMC.temperature_ladder_helpers as th
from DTMCMC.corr_summary_helpers import CorrelationSummary
from DTMCMC.dtmcmc_sampler import DTMCMCSampler
from DTMCMC.likelihood import RectangularLikelihood
from DTMCMC.proposal_manager import ProposalManager
from DTMCMC.temperature_ladder_helpers import GeometricTemperatureLadder

In [ ]:
# This is an example likelihood object for a gaussian likelihood
# In many cases this may be the only thing that needs to be modified to extend the sampler
# Subclassing RectangularLikelihood provides uniform prior draws, bounds handling,
# Fisher-matrix support, and plotting helpers for rectangular bounds automatically,
# so a new likelihood only has to supply its log likelihood


@njit()
def get_loglike(v):
    """Get an n dimensional gaussian likelihood"""
    const = np.log(1.0 / np.sqrt(2.0 * np.pi))  # normalization constant
    res = 0.0
    for itrp in range(v.shape[0]):
        res += const - 1 / 2 * v[itrp] ** 2
    return res


@njit(inline='always')
def _loglike_native(params_in, _state):
    """The same log likelihood with the (unused here) native state bundle argument."""
    return get_loglike(params_in)


class Likelihood(RectangularLikelihood):
    """class to manage the likelihood-specific essential functions for the sampler"""

    def __init__(self, n_par=100, cutoff=5.0):
        """Create the class and store any object specific variables"""
        super().__init__(n_par, np.full(n_par, -cutoff), np.full(n_par, cutoff))

    def get_loglike(self, params_in):
        """Get the log likelihood given a set of parameters v"""
        return get_loglike(params_in)

    def bind_native_loglike(self):
        """Opt into the compiled serial block kernel.

        This method is optional: delete it and the sampler transparently
        runs the same likelihood through the Python orchestration path
        (kernel_backend='auto' falls back silently)
        """
        return _loglike_native

In [ ]:
# starting variables
n_chain = 15  # number of total chains for parallel tempering
n_cold = 10  # number of T=1 chains for parallel tempering
n_burnin = 1000  # number of iterations to discard as burn in
block_size = 1000  # number of iterations per block when advancing the chain state
store_size = 30001  # number of samples to store total
N_blocks = store_size // block_size  # number of blocks the sampler must iterate through
n_par = 5

de_size = 1000  # number of samples to store in the differential evolution buffer
T_max = 100.0  # maximum temperature for geometric part of temperature ladder

params_true = np.zeros(n_par)  # true parameters for search

In [ ]:
# Configuration values for the proposal managers
# Any value not given falls back to the managers' built-in defaults;
# each manager documents its options in its StrategyParameters dataclass
config = configparser.ConfigParser()
config['ProposalManager'] = {}
config['FisherJumpManager'] = {'sigma_default': '100.0'}
config['DEJumpManager'] = {'de_size': str(de_size)}
config['AuxilliaryJumpManager'] = {}
config['PriorManager'] = {}

# The temperature ladder object
# By default, we use a geometric temperature ladder because it is standard
# The geometric ladder works particularly well when the likelihood is nearly Gaussian
# Other ladder designs may work better in other cases
T_ladder = GeometricTemperatureLadder(n_chain, n_cold=n_cold, T_max=T_max)

# Get the likelihood object
# Extending the sampler for new likelihoods may only have to modify the Likelihood object
# If the default jump managers work well; if they don't then the jumps may need to be modified also
like_obj = Likelihood(n_par)
# Make sure the specified starting paramaters are actually in bounds
params_true = like_obj.correct_bounds(params_true)

# create the starting samples
starting_samples = np.zeros((T_ladder.n_chain, like_obj.n_par))
for itrt in range(n_chain):
    starting_samples[itrt] = like_obj.prior_draw()

# Create the overarching proposal manager objects

# ExchangeManager controls different parallel tempering exchange proposal strategies:
# For example, SEQUENTIAL_TARGETS will start from the highest temperature chain
# And propose exchanges with the next highest chain.
# SEQUENTIAL_TARGETS is a common strategy and may be best if the temperature ladder covers a large range of likelihoods
# RANDOM_TARGETS proposes exchanges randomly betwen all pairs of chains, which can reduce strange inter-chain correlations
# RANDOM_TARGETS may be inefficent if the temperature ladder is very long
exchange_manager = eh.ExchangeManager(strategy=eh.SEQUENTIAL_TARGETS, track_full_exchanges=False)

# Get the various non-exchange jump manager objects
# Jump managers must follow the abstract class specification in JumpManager to work properly

# FisherJumpManager handles fisher matrix jumps and standard deviation jumps
fisher_manager = fm.FisherJumpManager(T_ladder, like_obj, starting_samples, config)

# DEJumpManager handles differential evolution jumps
de_manager = dm.DEJumpManager(T_ladder, like_obj, config)

# AuxiallaryJumpManager is blank and is intended to serve as a template for new jump types
auxilliary_manager = am.AuxilliaryJumpManager(T_ladder, like_obj, config)

# PriorManager manages priors draws
prior_manager = pm.PriorManager(T_ladder, like_obj, config)

# Tuple of all managers needed for the current strategy
managers = (fisher_manager, de_manager, auxilliary_manager, prior_manager)

# This is the overall ProposalManager object that manages how proposals are generated
# And knows how often to execute each proposal type
# ProposalManager itself is core infrastructure and usually should not need to be modified directly
# Instead, the list of jump managers given to ProposalManager can be edited or expanded
# ProposalManager should automatically adapt to new correctly implemented JumpManagers
proposal_manager = ProposalManager(T_ladder, like_obj, managers, exchange_manager, config)


# Create the sampler object that combines and executes the entire sampler architecture
# This runs the sampler and holds all the proposal managers, samples, and summary information
# The sampler object overall is designed to be extendable (for example, allowing modifications to how it outputs information)
# However, the sampler execution itself (e.g., advance_block_ptmcmc) is core infrastructure and usually should not need to be modified directly
# as that could break the sampler
# I anticipate that most user modifications should be in other places, such as the proposal managers
# kernel_backend='auto' (the default) compiles the whole block loop with numba
# when every component of the proposal graph provides native bindings,
# and falls back to the Python orchestration path otherwise
mcc = DTMCMCSampler(
    T_ladder,
    like_obj,
    block_size,
    store_size,
    proposal_manager=proposal_manager,
    starting_samples=starting_samples,
)

In [ ]:
# The main loop which actually advances the MCMC state
# Can be run multiple times; in this configuration doing so will overwrite past samples
# Overwriting samples can be desirable if the sampler has not yet burned in
# Or if aspects of the sampler are being written to a file to save RAM
mcc.advance_N_blocks(N_blocks)

# Note that the printout of the acceptance ratios is a useful way to assess chain performance
# acceptance ratios that are very high may indicate poor exploration or even drifting off the maximum
# while very low acceptance ratios may indicate poorly designed jumps and poor mixing

In [ ]:
# Generate some summary information about correlations
# These functions analyze how efficiently the sampler is generating effective samples
# We can estimate the efficiency both directly using a correlation analysis (called correlation efficiency in printouts)
# Or by using bootstrap variance estimates (called empirical efficiency in the printouts)
# When it works well, the correlation-based analysis is usually more precise
# But the boostrap-based analysis can be more robust sometimes, and so may be more accurate
# When both performance estimates agree they are likely capturing the behavior of the sampler well
# Note that depending on the likelihood structure performance can be highly dimension-dependent
corr_sum = CorrelationSummary()
corr_sum.summarize_blocks(mcc, mcc.tracker_manager, n_burnin)
corr_sum.final_prints(mcc, n_burnin)

# Print various kinds of diagnostic information about the parallel tempering performance
# An efficient temperature ladder can greatly improve sampler robustness
# And also improve effective sample generation rate
# Note that there can sometimes be a tradeoff between robustness and efficient sample generation
# This tradeoff is sometimes called exploration vs exploitation
dch.print_diagnostic_commentary(mcc)

# The sampler also keeps deterministic likelihood-evaluation accounting by source
print('likelihood evaluations by source:', mcc.eval_accounting)

# get flattened samples for plotting
samples_flattened, logLs_flattened = mcc.get_stored_flattened(
    corr_sum.restrict_n_burnin(mcc, n_burnin), n_chain_out=n_cold
)

In [ ]:
# A corner plot is a standard way to show the results of the sampler
# If the number of dimensions is large reducing the plot to a subspace may be necessary
import corner

# RectangularLikelihood provides default labels and sample formatting for plots;
# override get_labels/format_samples_output on the likelihood to customize them
samples_format, params_true_format = like_obj.format_samples_output(mcc.samples_store[:, 0, :], params_true)
labels = like_obj.get_labels()

# create the corner plot figure
fig = plt.figure(figsize=(10, 7.5))
figure = corner.corner(
    samples_format,
    fig=fig,
    bins=25,
    hist_kwargs={'density': True},
    show_titles=True,
    title_fmt=None,
    title_kwargs={'fontsize': 12},
    labels=labels,
    max_n_ticks=3,
    label_kwargs={'fontsize': 12},
    labelpad=0.15,
    smooth=0.25,
    levels=[0.682, 0.954],
)

# overplot the true parameters
corner.overplot_points(figure, params_true_format[None], marker='s', color='tab:blue', markersize=4)
corner.overplot_lines(figure, params_true_format, color='tab:blue')

# adjust the figure to fit the box better
fig.subplots_adjust(wspace=0.0, hspace=0.0, left=0.05, top=0.95, right=0.99, bottom=0.05)
for ax in figure.get_axes():
    ax.tick_params(which='both', direction='in', bottom=True, top=True, left=True, right=True, labelsize=6)
plt.show()

In [ ]:
# The autocorrelation length summarizes how correlated samples in a given chain are with each other
# We can compute the autocorrelation length for each dimension of the parameter space individually
# As well as the autocorrelation length for the likelihoods
# All else being equal, lower autocorrelation lengths always mean the sampler is performing better
# However, the autocorrelation can be artificially deflated if the sampler is missed some important structure completely
# If the sampler is exactly Markovian, autocorrelations should be positive semi-definite and monotonically decreasing
# (Up to numerical noise)
# Therefore the shape of autocorrelation plots can be an important diagnostic of poor overall performance
# And examining the autocorrelations for each dimension separately can give ideas for where customized jumps might be helpful
# Note: when there are multiple chains at the same temperature, the autocorrelation length does not tell the full story:
# Cross correlations between chains can also be important diagnostics of the overall performance of the sampler

# Make some autocorrelation plots
autocorrs = np.array(corr_sum.autocorr_lims).copy()  # get the unnormalized correlations
autocorr_use = np.max(corr_sum.est_vars_auto / autocorrs[:, 0])  # get the maximum autocorrelation length of
autocorrs = (autocorrs.T / autocorrs[:, 0]).T  # get the properly normalized autocorrelation

plt.plot(autocorrs.T)
plt.xlim(0, 3 * autocorr_use)
plt.xlabel('lag')
plt.title('Autocorrelation of parameters in cold chain for each dimension')
plt.show()

In [ ]:
# Average Likelihoods
# The spacings between the average likelihoods of the chains can be an indicator of disconnects
# Relatively consistent spacing is good: large gaps may indicate a disconnection
# Such disconnections can be caused by poorly sampled phase transitions
# Burn-in can be considered likely complete when all chains have stabilized at an equilibrium mean likelihood
# If the mean likelihood of an individual chain varies a lot over the course of the evolution it can indicate nonstationarity
# Heat capacity diagnostics below can help spot where to add more chains if necessary

plt.plot(mcc.logL_means)
plt.xlabel('Block Number')
plt.ylabel('<Log L>')
plt.title('Mean likelihood over time')
plt.show()

In [ ]:
# The heat capacity is an important predictor of the efficiency of the temperature ladder
# In analogy to thermodynamics, the 'Energy' of a state is -<log L>
# So the 'heat capacity', thermodynamically defined as C=d<E>/dT, can be defined analogously C=-d <log L>/dT
# However, this is an inefficient way to actually estimate the heat capacity.
# Using partition functions and thermodynamic assumptions, we can prove the following identity:
# d<E>/dT = Var(E)/T^2.
# Var(E)/T^2 turns out to be a greatly superior estimator of the heat capacity, as can be seen below

# Note that I calculate the derivative using a change of variables beta=1/T to handle T->Infinity properly
plt.semilogx(
    mcc.Ts[mcc.n_cold - 1 :],
    np.gradient(mcc.logL_means[-1][mcc.n_cold - 1 :], mcc.betas[mcc.n_cold - 1 :]) * mcc.betas[mcc.n_cold - 1 :] ** 2,
)
plt.semilogx(mcc.Ts[mcc.n_cold - 1 :], mcc.logL_vars[-1][mcc.n_cold - 1 :] / mcc.Ts[mcc.n_cold - 1 :] ** 2)
plt.legend(['Derivative Estimator', 'Variance Estimator'])
plt.xlabel('T')
plt.ylabel('C')
plt.title('Heat Capacity of MCMC Sampler')
plt.show()

In [ ]:
# Using the heat capacity
# we can predict what a more optimal temperature ladder
# with the same number of temperatures might have looked like

Ts_predict = th.entropy_spacing(n_chain, mcc.betas, mcc.logL_vars[-1])

plt.semilogy(mcc.Ts)
plt.semilogy(Ts_predict)
plt.xlabel('index')
plt.ylabel('T')
plt.title('Predicted improved temperature ladder')
plt.legend(['Original', 'Improved'])

In [ ]:
# What is differential evolution?
# Differential evolution is fundamentally a genetic algorithm
# That automatically learns the two-point correlation structure of a problem
# It is very good at finding widely separated modes, and can explore strangely shaped contours
# It can suffer in very high dimensional problems
# An advantage of combining it with parallel tempering is that it can learn
# the structure at multiple different temperatures at once

# Here is a visualization of what the differential evolution buffer learns

buffer_cold = de_manager.de_buffer[:, 0, :]
buffer_hot = de_manager.de_buffer[:, -1, :]
n_samples = 50

samples_sel = np.random.randint(0, buffer_cold.shape[0], n_samples)
buffer_samples_cold = buffer_cold[samples_sel]
buffer_samples_hot = buffer_hot[samples_sel]


for itrd1 in range(buffer_samples_cold.shape[0]):
    for itrd2 in range(buffer_samples_cold.shape[0]):
        if itrd1 != itrd2:
            diff_cold = buffer_samples_cold[itrd1] - buffer_samples_cold[itrd2]
            diff_hot = buffer_samples_hot[itrd1] - buffer_samples_hot[itrd2]

            plt.plot([0, diff_hot[0]], [0, diff_hot[1]], color='tab:red', alpha=0.1)
            plt.plot([0, diff_cold[0]], [0, diff_cold[1]], color='tab:blue', alpha=0.3)

plt.xlabel(r'$v_1$')
plt.ylabel(r'$v_2$')
plt.legend(['T=' + str(mcc.Ts[-2]), 'T=1'], loc=0)
plt.title('Visualization of differential evolution image of likelihood at different Ts')
plt.show()